# 📘 Module 06 — PyTorch from Scratch
## Chapter 6.3 — Building Neural Networks with `nn.Module`

> **Learning Path:** Zero to Hero — AI Engineer  
> **Module:** 06 of 18  
> **Topics:** Core Building Blocks (`nn.Linear`, `nn.ReLU`), Pipelines with `nn.Sequential`, Subclassing `nn.Module` (`__init__` & `forward`), Parameter Auditing, Counting & Layer Freezing  
> **Format:** Notes first → Code second  

---


## 🎯 What You Will Learn

By the end of this chapter, you will be able to:

- [ ] Explain how `nn.Linear` layers encapsulate learned weight and bias parameter tensors
- [ ] Rapidly prototype feedforward architectures using `nn.Sequential` containers
- [ ] Subclass `nn.Module` to build arbitrary deep architectures adhering to PyTorch OOP standards
- [ ] Audit model parameters: count trainable weights, inspect shapes, and track layer names
- [ ] Freeze specific layers with `param.requires_grad = False` for fine-tuning and transfer learning

**Estimated Reading Time:** ~30 minutes


## 🗂️ Table of Contents

1. [The Core Building Blocks — `nn.Linear` & Layer Internals](#1-the-core-building-blocks--nnlinear--layer-internals)
2. [Rapid Prototyping with `nn.Sequential`](#2-rapid-prototyping-with-nnsequential)
3. [The Universal Architecture Blueprint — Subclassing `nn.Module`](#3-the-universal-architecture-blueprint--subclassing-nnmodule)
4. [Parameter Auditing, Counting, & Layer Freezing](#4-parameter-auditing-counting--layer-freezing)
5. [Chapter Summary](#-chapter-summary)
6. [Resources & Further Reading](#-resources--further-reading)
7. [Next Chapter Preview](#-next-chapter-chapter-64--datasets-and-dataloaders)


---

## 1. The Core Building Blocks — `nn.Linear` & Layer Internals

---

### 🗣️ In Simple English

In **Module 05**, we manually initialized weight matrices $W$ and bias vectors $\mathbf{b}$, carefully setting their shapes and calculating $X @ W + \mathbf{b}$.

PyTorch packages this entire sequence into a single, pre-built class: **`nn.Linear(in_features, out_features)`**.
When you write:
```python
layer = nn.Linear(in_features=4, out_features=8)
```
PyTorch automatically:
1. Allocates a weight matrix $W$ with Kaiming/He uniform initialization.
2. Allocates a bias vector $\mathbf{b}$ initialized to small values.
3. Sets `requires_grad=True` on both parameters automatically.
4. Executes optimized BLAS matrix multiplications during the forward pass ($Y = X W^T + \mathbf{b}$).

The subtle design detail is: PyTorch stores the weight matrix internally with shape **`(out_features, in_features)`**, not `(in_features, out_features)`! It does this so that the mathematical operation computed is:
$$Y = X W^T + \mathbf{b}$$
This internal transposition aligns with C-contiguous GPU memory caching conventions.

---

### 🌍 Real-Life Example

**Scenario:** Building an automated customer credit risk assessment layer.

Suppose an applicant arrives with $4$ numerical financial features (Credit Score, Annual Income, Debt Ratio, Years Employed). We want to map these $4$ raw features into $3$ latent risk factor scores (Default Risk, Fraud Risk, Liquidity Risk).

| Layer Specification | Concrete Mathematical Entity | Tensor Shape | Number of Parameters |
|:---|:---|:---:|:---:|
| `in_features = 4` | Raw applicant features | `(N, 4)` | — |
| `out_features = 3` | Latent risk factor scores | `(N, 3)` | — |
| `layer.weight` | Synaptic connection strengths | `(3, 4)` | $3 \times 4 = 12$ weights |
| `layer.bias` | Threshold baseline offsets | `(3,)` | $3$ biases |
| **Total Parameters** | All trainable numbers in layer | — | $12 + 3 = \mathbf{15}$ parameters |

---

### 📖 How It Works (Step by Step)

1. **Instantiation:**
   ```python
   fc = nn.Linear(in_features=4, out_features=3, bias=True)
   ```
2. **The Forward Pass:**
   Passing an input batch $X \in \mathbb{R}^{N \times 4}$:
   ```python
   output = fc(X)  # Output shape: (N, 3)
   ```
   Under the hood, PyTorch computes:
   $$\text{output}_{ij} = \sum_{k=1}^4 X_{ik} W_{jk} + b_j$$
3. **Activation Layers:**
   PyTorch provides activations as both callable modules in `torch.nn` (`nn.ReLU()`, `nn.GELU()`, `nn.Sigmoid()`) and pure functional calls in `torch.nn.functional` (`F.relu(x)`).
   - Use `nn.ReLU()` when constructing `nn.Sequential` pipelines.
   - Use `F.relu(x)` inside custom `forward()` methods when activations have no trainable parameters.

---

### 🔑 Key Concepts

| Layer Type | PyTorch Class | Math Formula | Trainable Parameters? |
|:---|:---|:---:|:---:|
| **Dense / Linear** | `nn.Linear(in_f, out_f)` | $X W^T + \mathbf{b}$ | ✅ Yes ($W$ and $\mathbf{b}$) |
| **ReLU Activation** | `nn.ReLU()` | $\max(0, x)$ | ❌ No ($0$ parameters) |
| **GELU Activation** | `nn.GELU()` | $x \cdot \Phi(x)$ | ❌ No ($0$ parameters) |
| **Dropout** | `nn.Dropout(p=0.5)` | Mask with probability $p$ | ❌ No ($0$ parameters) |

---

### 🗺️ Visual Overview

```
THE NN.LINEAR(in_features=3, out_features=2) TRANSFORMATION:

  Input Batch X (N, 3)           Weight Matrix W.t() (3, 2)         Bias b (1, 2)            Output Y (N, 2)
  ┌─────────────────┐             ┌───────────────┐              ┌───────────────┐        ┌───────────────┐
  │ x_11  x_12  x_13│             │ w_11    w_21  │              │  b_1    b_2   │        │ y_11    y_12  │
N │  ...   ...   ...│     @     3 │ w_12    w_22  │      +       │  ...    ...   │   =  N │  ...     ...  │
  │ x_N1  x_N2  x_N3│             │ w_13    w_23  │              └───────────────┘        │ y_N1    y_N2  │
  └─────────────────┘             └───────────────┘                   (1, 2)              └───────────────┘
        (N, 3)                         (3, 2)                       Broadcast                   (N, 2)
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"If `in_features = 4` and `out_features = 8`, the weight tensor has shape `(4, 8)`."* | **False!** In PyTorch, `layer.weight.shape` is **`(8, 4)`** (`out_features, in_features`). This is an intentional convention so that matrix multiplication computes $X W^T + b$. |

---

### ✅ When to Use This

- Use `nn.Linear` for all fully connected layers, classification heads, and projection layers in Transformers.

---

### 📊 Section Quick Reference

| Attribute | Meaning | Inspection Syntax |
|:---|:---|:---|
| `layer.weight` | Trainable weight tensor | `layer.weight.shape` |
| `layer.bias` | Trainable bias tensor | `layer.bias.shape` |
| `layer.in_features` | Expected input dimension | `layer.in_features` |
| `layer.out_features` | Emitted output dimension | `layer.out_features` |

---


In [1]:
# ✅ Code Demo: Inspecting nn.Linear Layer Weights, Biases, and Forward Math
# Concept: Weight Transposition, Bias Broadcasting, and Parameter Initialization
# -----------------------------------------------------------------------------

import os
import torch
import torch.nn as nn

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-06-PyTorch-Foundations/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Instantiate a Linear Layer: 4 Inputs -> 3 Outputs
torch.manual_seed(42)
linear_layer = nn.Linear(in_features=4, out_features=3, bias=True)

print("=" * 65)
print("INSPECTING NN.LINEAR LAYER INTERNALS")
print("=" * 65)
print(f"Layer Class:        {linear_layer}")
print(f"Weight Tensor Shape: {linear_layer.weight.shape}  [out_features=3, in_features=4]")
print(f"Bias Tensor Shape:   {linear_layer.bias.shape}    [out_features=3]")
print(f"Weight requires_grad: {linear_layer.weight.requires_grad} (Auto-enabled!)")
print(f"Bias requires_grad:   {linear_layer.bias.requires_grad}")
print("-" * 65)

# 2. Forward Pass on a Batch of 2 Samples
X_batch = torch.tensor([
    [1.0, 2.0, 3.0, 4.0],
    [5.0, 6.0, 7.0, 8.0]
], dtype=torch.float32)

output_py = linear_layer(X_batch)

# 3. Manual Verification: Y = X @ W.T + b
manual_output = torch.matmul(X_batch, linear_layer.weight.t()) + linear_layer.bias

print("FORWARD PASS NUMERICAL VERIFICATION:")
print(f"Input Batch Shape:  {X_batch.shape} (N=2, d=4)")
print(f"Output Batch Shape: {output_py.shape} (N=2, c=3)")
print(f"PyTorch Output:\n{output_py.detach()}")
print(f"\nManual Math (X @ W.T + b):\n{manual_output.detach()}")
print(f"\nExact Match: {torch.allclose(output_py, manual_output)} ✅")
print("=" * 65)


INSPECTING NN.LINEAR LAYER INTERNALS
Layer Class:        Linear(in_features=4, out_features=3, bias=True)
Weight Tensor Shape: torch.Size([3, 4])  [out_features=3, in_features=4]
Bias Tensor Shape:   torch.Size([3])    [out_features=3]
Weight requires_grad: True (Auto-enabled!)
Bias requires_grad:   True
-----------------------------------------------------------------
FORWARD PASS NUMERICAL VERIFICATION:
Input Batch Shape:  torch.Size([2, 4]) (N=2, d=4)
Output Batch Shape: torch.Size([2, 3]) (N=2, c=3)
PyTorch Output:
tensor([[3.0675, 0.6042, 1.6264],
        [7.6253, 0.7705, 4.0349]])

Manual Math (X @ W.T + b):
tensor([[3.0675, 0.6042, 1.6264],
        [7.6253, 0.7705, 4.0349]])

Exact Match: True ✅


---

## 2. Rapid Prototyping with `nn.Sequential`

---

### 🗣️ In Simple English

If your neural network is a straightforward assembly line where data passes sequentially from Layer 1 $\to$ ReLU $\to$ Layer 2 $\to$ Softmax with no side-branches or complex routing, you don't even need to write a class!

You can use **`nn.Sequential`**.

Think of it this way: Imagine a conveyor belt at an airport luggage scanner:
- Station 1: X-ray scanner (`nn.Linear`)
- Station 2: Metal detector (`nn.ReLU`)
- Station 3: Sorting arm (`nn.Linear`)

`nn.Sequential` packages all these stations into a single unified object. You feed data into the front of the conveyor belt, and it automatically cascades through every layer in order and pops out the final predictions!

---

### 📖 How It Works (Step by Step)

#### Constructing an `nn.Sequential` Model
```python
model = nn.Sequential(
    nn.Linear(784, 128),  # Input layer (784 features -> 128 hidden)
    nn.ReLU(),            # Non-linear activation
    nn.Linear(128, 64),   # Hidden layer (128 -> 64)
    nn.ReLU(),            # Non-linear activation
    nn.Linear(64, 10)     # Output classification layer (64 -> 10 classes)
)
```

#### Ordered Dictionary Formulation (Giving Layers Names!)
By default, layers inside `nn.Sequential` are indexed by integers (`model[0]`, `model[1]`). You can assign human-readable names using `collections.OrderedDict`:
```python
from collections import OrderedDict

model = nn.Sequential(OrderedDict([
    ('hidden_1', nn.Linear(784, 128)),
    ('act_1',    nn.ReLU()),
    ('hidden_2', nn.Linear(128, 64)),
    ('act_2',    nn.ReLU()),
    ('classifier', nn.Linear(64, 10))
]))

# Now you can access layers by name!
print(model.classifier.weight.shape)
```

---

### 🔑 Key Concepts

| Feature | When `nn.Sequential` Shines | When `nn.Sequential` Fails |
|:---|:---|:---|
| **Architecture Topology** | Single-input, single-output linear cascades. | Multi-input networks, multi-output heads, skip connections (ResNet), attention blocks. |
| **Code Verbosity** | Extremely concise (3 lines of code). | Requires full custom class for branching logic. |

---

### 🗺️ Visual Overview

```
DATA FLOW THROUGH NN.SEQUENTIAL:

Input X ──► [ nn.Linear(784, 128) ] ──► [ nn.ReLU() ] ──► [ nn.Linear(128, 10) ] ──► Predictions
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"Can I build a ResNet (Skip Connection: $y = f(x) + x$) using `nn.Sequential` alone?"* | **No!** `nn.Sequential` has no mechanism to remember past inputs or add a parallel residual branch. Any architecture with skip connections or recurrent loops must be written as a custom `nn.Module` class. |

---

### ✅ When to Use This

- Use `nn.Sequential` for simple multi-layer perceptrons, small sub-blocks within larger architectures, or rapid baseline prototyping.

---

### 📊 Section Quick Reference

| Syntax | Action |
|:---|:---|
| `model = nn.Sequential(...)` | Creates sequential feedforward pipeline |
| `model[0]` | Accesses first layer by index |
| `model.layer_name` | Accesses layer by name (if using `OrderedDict`) |

---


In [2]:
# ✅ Code Demo: Building Feedforward Pipelines with nn.Sequential and OrderedDict
# Concept: Chaining Layers, Named Layers via OrderedDict, and Forward Evaluation
# -----------------------------------------------------------------------------

import os
from collections import OrderedDict
import torch
import torch.nn as nn

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-06-PyTorch-Foundations/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Standard nn.Sequential (Indexed by integers)
torch.manual_seed(42)
mlp_simple = nn.Sequential(
    nn.Linear(8, 16),
    nn.ReLU(),
    nn.Linear(16, 4)
)

print("=" * 65)
print("RAPID PROTOTYPING WITH NN.SEQUENTIAL")
print("=" * 65)
print("Simple Sequential Model Representation:\n", mlp_simple)
print(f"\nAccessing Layer 0: {mlp_simple[0]}")
print(f"Layer 0 Weights:   {mlp_simple[0].weight.shape}")
print("-" * 65)

# 2. Named nn.Sequential with OrderedDict
mlp_named = nn.Sequential(OrderedDict([
    ('input_projection',  nn.Linear(8, 16)),
    ('activation_relu',   nn.ReLU()),
    ('hidden_dense',      nn.Linear(16, 16)),
    ('activation_gelu',   nn.GELU()),
    ('classifier_head',   nn.Linear(16, 2))
]))

print("Named Sequential Model with OrderedDict:\n", mlp_named)
print(f"\nDirect Access to Classifier: {mlp_named.classifier_head}")
print(f"Classifier Output Dimension: {mlp_named.classifier_head.out_features}")
print("-" * 65)

# 3. Test Forward Pass on a Synthetic Batch of 3 Samples
test_input = torch.randn(3, 8)
predictions = mlp_named(test_input)
print(f"Input Shape:       {test_input.shape}")
print(f"Predictions Shape: {predictions.shape} (N=3 samples x 2 class logits)")
print(f"Output Logits:\n{predictions.detach()}")
print("=" * 65)


RAPID PROTOTYPING WITH NN.SEQUENTIAL
Simple Sequential Model Representation:
 Sequential(
  (0): Linear(in_features=8, out_features=16, bias=True)
  (1): ReLU()
  (2): Linear(in_features=16, out_features=4, bias=True)
)

Accessing Layer 0: Linear(in_features=8, out_features=16, bias=True)
Layer 0 Weights:   torch.Size([16, 8])
-----------------------------------------------------------------
Named Sequential Model with OrderedDict:
 Sequential(
  (input_projection): Linear(in_features=8, out_features=16, bias=True)
  (activation_relu): ReLU()
  (hidden_dense): Linear(in_features=16, out_features=16, bias=True)
  (activation_gelu): GELU(approximate='none')
  (classifier_head): Linear(in_features=16, out_features=2, bias=True)
)

Direct Access to Classifier: Linear(in_features=16, out_features=2, bias=True)
Classifier Output Dimension: 2
-----------------------------------------------------------------
Input Shape:       torch.Size([3, 8])
Predictions Shape: torch.Size([3, 2]) (N=3 sampl

---

## 3. The Universal Architecture Blueprint — Subclassing `nn.Module`

---

### 🗣️ In Simple English

When you want to build professional neural networks with custom architectures, skip connections, multi-head attention, or dynamic logic, you write a **Custom Class that inherits from `nn.Module`**.

Think of `nn.Module` as a magical template. By inheriting from it:
- PyTorch automatically registers any child layers you declare in `__init__()`.
- PyTorch tracks all their weights and biases automatically.
- Calling `model.to('cuda')` moves every single layer in your class to the GPU in one line.
- Calling `model.parameters()` collects every weight in the network to hand to the optimizer.

There are only **TWO golden rules** to building an `nn.Module`:
1. In `__init__()`: Always call `super().__init__()` first, then declare your layers.
2. In `forward()`: Define how input $x$ transforms as it flows through the layers.

---

### 🌍 Real-Life Example

**Scenario:** A multi-branch medical AI analyzing both tabular patient blood work and a diagnostic lung X-ray simultaneously.

```
Tabular Biomarkers ──► [ Dense Branch (nn.Linear) ] ───\
                                                        ──► [ Concatenate ] ──► Final Diagnosis Head
Chest X-Ray Image  ──► [ ConvNet Branch (CNN)     ] ───/
```
Such multi-modal architectures are impossible with simple sequential pipelines. Subclassing `nn.Module` makes this trivial!

---

### 📖 How It Works (Step by Step)

#### The Canonical Blueprint
```python
import torch
import torch.nn as nn
import torch.nn.functional as F

class CustomNeuralNetwork(nn.Module):
    def __init__(self, in_features, hidden_dim, out_classes):
        # RULE 1: Initialize parent class!
        super().__init__()
        
        # Instantiate sub-layers
        self.fc1 = nn.Linear(in_features, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, out_classes)
        self.dropout = nn.Dropout(p=0.2)

    def forward(self, x):
        # RULE 2: Define computation graph
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        logits = self.out(x)
        return logits
```

#### The Cardinal Rule of Forward Execution
> **NEVER call `model.forward(x)` directly in your code!**
> **ALWAYS call `model(x)`!**

When you write `model(x)`, Python invokes `nn.Module.__call__()`. Under the hood, PyTorch:
1. Executes forward pre-hooks (logging, telemetry).
2. Sets up autograd computation graph nodes.
3. Invokes your `forward(x)` method.
4. Executes forward post-hooks and gradient profilers.
If you call `model.forward(x)` directly, all PyTorch hooks and internal machinery are bypassed!

---

### 🔑 Key Concepts

| Method | Role | When It Runs |
|:---|:---|:---|
| `__init__()` | Architectural setup: defines layers, allocates weights. | Runs once when the model is created. |
| `forward(x)` | Dynamic graph execution: defines computation path. | Runs every time data passes through `model(x)`. |
| `super().__init__()`| Connects model to PyTorch C++ backend registration. | Must be the very first line of `__init__()`. |

---

### 🗺️ Visual Overview

```
THE NN.MODULE ARCHITECTURAL LIFECYCLE:

  class MyModel(nn.Module):
      def __init__():
          super().__init__()
          self.layer1 = nn.Linear(...)  ───► PyTorch registers layer1 in internal _modules dict!
          self.layer2 = nn.Linear(...)  ───► PyTorch registers layer2 in internal _modules dict!

      def forward(x):                   ───► Called automatically when writing `model(x)`!
          return self.layer2(F.relu(self.layer1(x)))
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"If I store layers in a standard Python list like `self.layers = [nn.Linear(...), nn.Linear(...)]`, PyTorch will register them."* | **A famous trap!** PyTorch does NOT inspect standard Python lists. Tensors inside a normal list will **NOT** appear in `model.parameters()` and will **NOT** move to GPU with `model.to('cuda')`! You must wrap them in **`nn.ModuleList`** (`self.layers = nn.ModuleList([...])`). |

---

### ✅ When to Use This

- Subclass `nn.Module` for any custom neural network with more than 2 layers, residual connections, or non-trivial data routing.

---

### 📊 Section Quick Reference

| Action | Correct Syntax | Incorrect Syntax |
|:---|:---|:---|
| **Parent init** | `super().__init__()` | Forgetting to call super |
| **Forward pass execution** | `output = model(x)` | `output = model.forward(x)` ❌ |
| **List of layers** | `nn.ModuleList([layer1, layer2])` | `[layer1, layer2]` ❌ |

---


In [3]:
# ✅ Code Demo: Building a Custom nn.Module Classifier with Skip Connections
# Concept: Subclassing nn.Module, super().__init__(), and Non-Linear Forward Pass
# -------------------------------------------------------------------------------

import os
import torch
import torch.nn as nn
import torch.nn.functional as F

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-06-PyTorch-Foundations/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Define Custom Architecture with Residual Skip-Connection
class ResidualMLP(nn.Module):
    def __init__(self, in_features=10, hidden_dim=32, out_classes=3):
        super().__init__()
        
        # Branch A: First dense projection
        self.fc1 = nn.Linear(in_features, hidden_dim)
        
        # Branch B: Non-linear hidden transformation
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        
        # Classifier output head
        self.out_head = nn.Linear(hidden_dim, out_classes)
        self.dropout = nn.Dropout(p=0.1)

    def forward(self, x):
        # Layer 1
        h1 = F.relu(self.fc1(x))
        
        # Layer 2 with RESIDUAL SKIP CONNECTION: h2 = Layer2(h1) + h1
        h2 = F.relu(self.fc2(h1)) + h1  # <-- Skip connection!
        h2 = self.dropout(h2)
        
        # Output logits
        logits = self.out_head(h2)
        return logits

# 2. Instantiate and Inspect Model
torch.manual_seed(42)
custom_model = ResidualMLP(in_features=10, hidden_dim=32, out_classes=3)

print("=" * 65)
print("CUSTOM NN.MODULE WITH SKIP CONNECTION")
print("=" * 65)
print("Model Structure:\n", custom_model)
print("-" * 65)

# 3. Verify Forward Pass (Using model(x), NEVER model.forward(x))
X_dummy = torch.randn(4, 10)  # Batch of 4 samples, 10 features
logits = custom_model(X_dummy) # Correct invocation!

print(f"Input Tensor Shape:  {X_dummy.shape}")
print(f"Output Logits Shape: {logits.shape} (N=4 samples, 3 class logits)")
print(f"Output Values:\n{logits.detach().round(decimals=3)}")
print("=" * 65)


CUSTOM NN.MODULE WITH SKIP CONNECTION
Model Structure:
 ResidualMLP(
  (fc1): Linear(in_features=10, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=32, bias=True)
  (out_head): Linear(in_features=32, out_features=3, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
)
-----------------------------------------------------------------
Input Tensor Shape:  torch.Size([4, 10])
Output Logits Shape: torch.Size([4, 3]) (N=4 samples, 3 class logits)
Output Values:
tensor([[ 0.0350, -0.3100, -0.2680],
        [ 0.1720, -0.4470,  0.0630],
        [ 0.0650, -0.3220,  0.1190],
        [ 0.4050, -0.0860, -0.4270]])


---

## 4. Parameter Auditing, Counting, & Layer Freezing

---

### 🗣️ In Simple English

When you train a neural network, how do you know how big it is? And what if you want to take a pre-trained model (like BERT or ResNet) and only train the **final output layer** while keeping the rest of the model locked in ice?

PyTorch makes this effortless through **Parameter Management**:
1. **Auditing:** You can inspect every weight and bias tensor by name using `model.named_parameters()`.
2. **Counting:** You can count every single floating-point parameter with a simple generator expression.
3. **Layer Freezing:** You can freeze any layer by setting `param.requires_grad = False`.

When a layer is frozen:
- Its weights are locked in place.
- Autograd **skips computing gradients for that layer** during `.backward()`.
- You save massive GPU VRAM and computation time!

---

### 🌍 Real-Life Example

**Scenario:** Fine-tuning an 86-million parameter pre-trained Vision Transformer on a specialized medical dataset of only 500 skin lesion biopsies.

If you train all 86 million weights on just 500 biopsies, the model will instantly overfit and memorize the noise.

| Strategy | Trainable Parameters | Risk of Overfitting | Training Speed |
|:---|:---:|:---:|:---:|
| **Train Full Model** | $86,000,000$ weights | 🚨 Catastrophic Overfitting | Slow (Requires 40GB VRAM) |
| **Freeze Backbone, Train Head** | **$2,000$ weights** (Frozen backbone) | 🛡️ Minimum Overfitting | **$10\times$ faster** (Fits in 6GB VRAM) |

---

### 📖 How It Works (Step by Step)

#### A. Counting Model Parameters
```python
# Total parameters
total_params = sum(p.numel() for p in model.parameters())

# Trainable parameters (requires_grad == True)
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total: {total_params:,} | Trainable: {trainable_params:,}")
```

#### B. Freezing Specific Layers
To freeze a layer, loop over its parameters and set `requires_grad = False`:
```python
# Freeze the first dense layer
for param in model.fc1.parameters():
    param.requires_grad = False
```

#### C. Passing Only Trainable Parameters to the Optimizer
```python
# Filter out frozen parameters so optimizer doesn't waste memory tracking them!
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.001
)
```

---

### 🔑 Key Concepts

| Method | Return Value | Purpose |
|:---|:---|:---|
| `model.parameters()` | Iterator of all tensors in model. | Handed directly to the optimizer. |
| `model.named_parameters()` | Iterator of `(name, tensor)` pairs. | Useful for auditing, logging, and selective layer freezing. |
| `p.requires_grad = False` | Disables gradient tracking for parameter `p`. | **Freezes** parameter during training. |
| `p.numel()` | Total number of elements in parameter `p`. | Used to compute model parameter count. |

---

### 🗺️ Visual Overview

```
LAYER FREEZING IN TRANSFER LEARNING:

  Input X ──► [ Layer 1: FROZEN (requires_grad=False) ]   ──► Weights locked in ice! No gradients computed!
                    │
                    ▼
              [ Layer 2: FROZEN (requires_grad=False) ]   ──► Feature extractor preserved!
                    │
                    ▼
              [ Output Head: ACTIVE (requires_grad=True) ] ──► ONLY this layer learns and updates!
```

---

### 🚫 Common Misconceptions

| ❌ Myth | ✅ Reality |
|:---|:---|
| *"If I set `param.requires_grad = False`, will forward pass activations still flow through that layer?"* | **Yes, 100%!** Freezing a layer does NOT stop data from passing forward. It simply prevents PyTorch from computing and storing gradients for that layer during the backward pass. |

---

### ✅ When to Use This

- Always print parameter counts when initializing models to catch accidental dimension blowups.
- Freeze early layers when fine-tuning pre-trained models on small datasets.

---

### 📊 Section Quick Reference

| Task | Code Snippet |
|:---|:---|
| **Total parameter count** | `sum(p.numel() for p in model.parameters())` |
| **Freeze specific layer** | `for p in model.layer.parameters(): p.requires_grad = False` |
| **Unfreeze all layers** | `for p in model.parameters(): p.requires_grad = True` |

---


In [4]:
# ✅ Code Demo: Parameter Auditing, Counting, and Selective Layer Freezing
# Concept: Inspecting Named Parameters, Calculating Model Size, and Verifying Freezing
# ------------------------------------------------------------------------------------

import os
import torch
import torch.nn as nn

# Ensure local asset path exists
assets_dir = 'assets' if os.path.exists('assets') else 'Module-06-PyTorch-Foundations/assets'
os.makedirs(assets_dir, exist_ok=True)

# 1. Instantiate a 3-Layer Neural Network
class MultiStageClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.feature_extractor = nn.Linear(20, 64)
        self.intermediate_dense = nn.Linear(64, 32)
        self.classification_head = nn.Linear(32, 2)
        
    def forward(self, x):
        x = torch.relu(self.feature_extractor(x))
        x = torch.relu(self.intermediate_dense(x))
        return self.classification_head(x)

model = MultiStageClassifier()

# 2. Audit Named Parameters
print("=" * 70)
print("AUDITING MODEL PARAMETERS & SIZES")
print("=" * 70)
print(f"{'Parameter Name':<32} | {'Shape':<16} | {'Count':<8} | {'Requires Grad'}")
print("-" * 70)
for name, p in model.named_parameters():
    print(f"{name:<32} | {str(list(p.shape)):<16} | {p.numel():<8} | {p.requires_grad}")

total_count = sum(p.numel() for p in model.parameters())
trainable_count = sum(p.numel() for p in model.parameters() if p.requires_grad)
print("-" * 70)
print(f"Total Parameters:     {total_count:,}")
print(f"Trainable Parameters: {trainable_count:,}")
print("=" * 70)

# 3. Selective Layer Freezing (Simulating Transfer Learning)
print("\n[Action]: FREEZING Feature Extractor & Intermediate Dense Layers...")
for p in model.feature_extractor.parameters():
    p.requires_grad = False
for p in model.intermediate_dense.parameters():
    p.requires_grad = False

# Re-evaluate parameter audit
trainable_after_freeze = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable Parameters After Freezing: {trainable_after_freeze:,} / {total_count:,} ({trainable_after_freeze/total_count*100:.1f}%)")

# 4. Proving Selective Backward Pass (Only Head Receives Gradients)
X_test = torch.randn(4, 20)
out = model(X_test).sum()
out.backward()

print("\nGRADIENT VERIFICATION AFTER BACKWARD PASS:")
print(f"  feature_extractor.weight.grad:   {model.feature_extractor.weight.grad} (Zero gradients computed! ✅)")
print(f"  classification_head.weight.grad: Shape={model.classification_head.weight.grad.shape} (Gradients computed! ✅)")
print("=" * 70)


AUDITING MODEL PARAMETERS & SIZES
Parameter Name                   | Shape            | Count    | Requires Grad
----------------------------------------------------------------------
feature_extractor.weight         | [64, 20]         | 1280     | True
feature_extractor.bias           | [64]             | 64       | True
intermediate_dense.weight        | [32, 64]         | 2048     | True
intermediate_dense.bias          | [32]             | 32       | True
classification_head.weight       | [2, 32]          | 64       | True
classification_head.bias         | [2]              | 2        | True
----------------------------------------------------------------------
Total Parameters:     3,490
Trainable Parameters: 3,490

[Action]: FREEZING Feature Extractor & Intermediate Dense Layers...
Trainable Parameters After Freezing: 66 / 3,490 (1.9%)

GRADIENT VERIFICATION AFTER BACKWARD PASS:
  feature_extractor.weight.grad:   None (Zero gradients computed! ✅)
  classification_head.weight.gra

---

## 📝 Chapter Summary

| Component | Plain English Meaning | PyTorch Syntax | Best Practice |
|:---|:---|:---|:---|
| **`nn.Linear`** | Dense fully connected layer | `nn.Linear(in_dim, out_dim)` | Internal weight shape is `(out_dim, in_dim)`. |
| **`nn.Sequential`** | Sequential layer conveyor belt | `nn.Sequential(...)` | Use for fast linear pipelines; avoid for branching models. |
| **`nn.Module`** | Foundational blueprint for custom models | `class MyModel(nn.Module):` | Always call `super().__init__()` and implement `forward()`. |
| **Model Invocation** | Executing forward pass with hooks | `output = model(x)` | **Never** call `model.forward(x)` directly. |
| **Parameter Count** | Number of learned weights | `sum(p.numel() for p in model.parameters())` | Audit parameters to catch accidental memory bloat. |
| **Layer Freezing** | Locking parameters in ice | `p.requires_grad = False` | Essential for fine-tuning pre-trained backbones. |

---


## 🔗 Resources & Further Reading

| Resource | Type | Why Read It |
|:---|:---|:---|
| **[PyTorch nn.Module Official Documentation](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)** | Official Docs | The master reference for all hooks, buffers, state dicts, and module registration. |
| **[PyTorch Neural Network Tutorial (torch.nn)](https://pytorch.org/tutorials/beginner/nn_tutorial.html)** | Jeremy Howard / PyTorch | Outstanding tutorial breaking down how `torch.nn` abstractions evolved from pure math. |
| **[ModuleList vs Sequential in PyTorch](https://discuss.pytorch.org/t/when-should-i-use-nn-modulelist-and-when-should-i-use-nn-sequential/20363)** | Technical Discussion | Deep discussion on container selection and module parameter discovery. |

---


## ➡️ Next Chapter: Chapter 6.4 — Datasets and DataLoaders

Now that we can construct custom neural architectures, how do we feed them millions of real-world training examples efficiently without crashing RAM?
- **The `Dataset` Class:** Building custom datasets by implementing `__len__()` and `__getitem__()`.
- **The `DataLoader` Pipeline:** Multi-threaded parallel worker streaming, batching, and shuffling.
- **Image Transformations:** Normalization, tensor conversion, and data augmentation.
- **Handling Real Data:** Streaming handwritten digits directly from local workspace storage!

---
*Made with ❤️ | Zero to Hero — AI Engineer Learning Path*
